In [1]:
import numpy as np
import random

# -------------------------------
# Step 1: Define the Environment (Line World)
# -------------------------------
class LineWorld:
    def __init__(self, num_states=7):
        """
        States: [0, 1, 2, 3, 4, 5, 6]
        0 and 6 are Terminal States.
        Start state is 3 (the exact middle).
        """
        self.num_states = num_states
        self.start_state = num_states // 2
        self.state = self.start_state

    def reset(self):
        self.state = self.start_state
        return self.state

    def step(self, action):
        """
        Actions: 0 = Left, 1 = Right
        Rewards: +1 if reaching state 6 (Right Goal), -1 if reaching state 0 (Left Trap), 0 otherwise.
        """
        if action == 0:
            self.state -= 1
        elif action == 1:
            self.state += 1

        # Check if terminal
        if self.state == 0:
            return self.state, -1.0, True
        elif self.state == self.num_states - 1:
            return self.state, 1.0, True
        else:
            return self.state, 0.0, False

# -------------------------------
# Step 2: Define a Random/Uniform Policy
# -------------------------------
def policy(state):
    # Equal 50/50 probability to go Left (0) or Right (1)
    return random.choice([0, 1])

# -------------------------------
# Step 3: Generate an Episode
# -------------------------------
def generate_episode(env):
    """
    Runs a single episode from start to termination using the policy.
    Returns a list of tuples: (state, action, reward)
    """
    episode = []
    state = env.reset()
    done = False

    while not done:
        action = policy(state)
        next_state, reward, done = env.step(action)
        episode.append((state, action, reward))
        state = next_state

    return episode

# -------------------------------
# Step 4: First-Visit Monte Carlo Prediction
# -------------------------------
def monte_carlo_prediction(env, num_episodes, gamma=1.0):
    """
    Estimates the state-value function V(s) for the uniform random policy.
    """
    # Initialize value table and an empty tracking list for returns
    V = np.zeros(env.num_states)
    returns = {s: [] for s in range(env.num_states)}

    for ep in range(num_episodes):
        episode = generate_episode(env)
        G = 0 # Cumulative Return

        # Traverse the episode backwards to compute returns easily
        visited_states = set()

        # Step backwards through the episode: t = T-1, T-2, ..., 0
        for t in reversed(range(len(episode))):
            state, action, reward = episode[t]
            G = gamma * G + reward

            # First-Visit Check: Only update if this is the first time
            # visiting 'state' in this specific episode
            state_history = [step[0] for step in episode[:t]]
            if state not in state_history:
                returns[state].append(G)
                V[state] = np.mean(returns[state])

    return V

# -------------------------------
# Step 5: Run Evaluation
# -------------------------------
env = LineWorld(num_states=7)
num_episodes = 5000
estimated_values = monte_carlo_prediction(env, num_episodes=num_episodes)

print(f"Estimated State Values V(s) after {num_episodes} episodes:")
for s in range(env.num_states):
    if s == 0:
        print(f"State {s} (Terminal Trap) : {estimated_values[s]:.4f}")
    elif s == env.num_states - 1:
        print(f"State {s} (Terminal Goal) : {estimated_values[s]:.4f}")
    else:
        print(f"State {s}                 : {estimated_values[s]:.4f}")


Estimated State Values V(s) after 5000 episodes:
State 0 (Terminal Trap) : 0.0000
State 1                 : -0.6648
State 2                 : -0.3426
State 3                 : -0.0072
State 4                 : 0.3358
State 5                 : 0.6719
State 6 (Terminal Goal) : 0.0000
